# EasyRAG Latency Cost And Regression Checks

## What you will do

- collect latency and prompt-size proxies from existing metadata
- build a lightweight baseline snapshot
- compare a changed configuration against that baseline as a regression check

## Why this matters

EasyRAG does not yet expose a formal cost API, but it already exposes enough timing and context metadata to build useful operational checks.


## Source code anchors

- `easyrag.rag.retrieval.pipeline.execute_query`
- `easyrag.rag.generation.pipeline.generate_answer`
- `easyrag.rag.types.AnswerParam`
- `easyrag.rag.types.QueryParam`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)


## Deterministic path

We will treat `stage_timings_ms`, retrieval-query count, selected citation count, and context size as operational proxies. They are not a billing API, but they are real runtime signals the code already gives us.


In [ ]:
ops_tmp = tempfile.TemporaryDirectory()
ops_rag = EasyRAG(
    working_dir=ops_tmp.name,
    workspace="ops-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(ops_rag.initialize_storages())
run_sync(
    ops_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses query rewrite for grounded retrieval.\n",
            "# Policy\nCitation-aware answers preserve traceability.\n",
        ],
        ids=["doc::retrieval", "doc::policy"],
        file_paths=["docs/retrieval.md", "docs/policy.md"],
    )
)
question = "How does EasyRAG keep answers grounded?"
baseline_query_param = QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=2)
baseline_answer_param = AnswerParam(max_citations=2, max_context_chars=150)
changed_query_param = QueryParam(mode="hybrid", rewrite_enabled=True, mqe_enabled=True, mqe_variants=3, chunk_top_k=4)
changed_answer_param = AnswerParam(max_citations=3, max_context_chars=260)

baseline_answer = run_sync(ops_rag.aanswer(question, baseline_query_param, baseline_answer_param))
changed_answer = run_sync(ops_rag.aanswer(question, changed_query_param, changed_answer_param))

def snapshot(name, answer_result):
    retrieval_metadata = answer_result.metadata["retrieval_metadata"]
    return {
        "name": name,
        "total_retrieval_ms": retrieval_metadata["stage_timings_ms"]["total"],
        "retrieval_query_count": len(retrieval_metadata["retrieval_queries"]),
        "selected_citation_count": answer_result.metadata["selected_citation_count"],
        "context_chars": answer_result.metadata["context_chars"],
        "vector_backend": retrieval_metadata["vector_backend"],
    }

baseline_snapshot = snapshot("baseline", baseline_answer)
changed_snapshot = snapshot("changed", changed_answer)
_print_json({"baseline": baseline_snapshot, "changed": changed_snapshot})


## Output inspection

The next cell turns those snapshots into a regression check. The thresholds are notebook-local policy choices built on top of real runtime metadata.


In [ ]:
regression_report = {
    "retrieval_latency_regressed": changed_snapshot["total_retrieval_ms"] > baseline_snapshot["total_retrieval_ms"] * 2.5,
    "query_count_regressed": changed_snapshot["retrieval_query_count"] > baseline_snapshot["retrieval_query_count"] + 3,
    "context_budget_regressed": changed_snapshot["context_chars"] > baseline_snapshot["context_chars"] + 150,
}
_print_json(regression_report)


## Provider-backed path

If providers are configured, the same snapshot pattern works with provider-backed retrieval. The notebook still inspects metadata rather than inventing a separate cost interface.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-ops-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(provider_rag.ainsert("# Retrieval\nGrounded retrieval keeps answers inspectable.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
        provider_answer = run_sync(provider_rag.aanswer("How do answers stay inspectable?", QueryParam(mode="hybrid", chunk_top_k=2), AnswerParam()))
        _print_json({
            "retrieval_metadata": provider_answer.metadata["retrieval_metadata"],
            "context_chars": provider_answer.metadata["context_chars"],
        })
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

Operational evaluation here is built from proxies, but they are real proxies from the runtime. More query variants usually increase latency. Larger citation budgets usually increase context size. Regression checks become useful when they compare like-for-like snapshots instead of relying on vague impressions.


In [ ]:
run_sync(ops_rag.finalize_storages())
ops_tmp.cleanup()
print("Cleaned up the latency-and-regression workspace.")


## Next steps

- Continue with [07-optimization-overview.md](../../docs/07-optimization-overview.md) to turn these checks into optimization priorities.
- Read [06-evaluation-overview.md](../../docs/06-evaluation-overview.md) for the broader evaluation map.
- Revisit [06_05_eval_driven_debugging.ipynb](06_05_eval_driven_debugging.ipynb) if you want the correctness-first debugging loop before the operational layer.
